In [1]:
!pip install -q -U earthengine-api geemap scikit-learn xgboost joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.5/481.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 50.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.56.3 which is incompatible.


In [2]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier
import joblib

In [3]:
ee.Authenticate()

ee.Initialize(
    project='fourth-groove-470214-u2'
)

print("Earth Engine initialized successfully!")

Earth Engine initialized successfully!


In [7]:
study_area = ee.Geometry.Rectangle([
    78.20, 17.25,
    78.65, 17.60
])

print("Study area created!")

Study area created!


In [4]:
def mask_s2_clouds(image):
    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return image.updateMask(mask).divide(10000)

In [5]:
def get_sentinel_composite(start_date, end_date, region):

    collection = (
        ee.ImageCollection(
            'COPERNICUS/S2_SR_HARMONIZED'
        )
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                'CLOUDY_PIXEL_PERCENTAGE',
                20
            )
        )
        .map(mask_s2_clouds)
    )

    print(
        start_date,
        "to",
        end_date,
        "| Images:",
        collection.size().getInfo()
    )

    return collection.median().clip(region)

In [8]:
image_2020 = get_sentinel_composite(
    '2020-01-01',
    '2020-01-31',
    study_area
)

image_2025 = get_sentinel_composite(
    '2025-01-01',
    '2025-01-31',
    study_area
)

2020-01-01 to 2020-01-31 | Images: 8
2025-01-01 to 2025-01-31 | Images: 7


In [9]:
def add_spectral_indices(image):

    ndvi = image.normalizedDifference(
        ['B8', 'B4']
    ).rename('NDVI')

    ndwi = image.normalizedDifference(
        ['B3', 'B8']
    ).rename('NDWI')

    ndbi = image.normalizedDifference(
        ['B11', 'B8']
    ).rename('NDBI')

    return image.addBands([
        ndvi,
        ndwi,
        ndbi
    ])

In [10]:
features_2020 = add_spectral_indices(image_2020)
features_2025 = add_spectral_indices(image_2025)

In [11]:
feature_bands = [
    'B2',
    'B3',
    'B4',
    'B8',
    'B11',
    'B12',
    'NDVI',
    'NDWI',
    'NDBI'
]

print(feature_bands)

['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI']


## using world cover as a reference for training on base models for labels are they are supervised
Sentinel-2 2020 + WorldCover 2020  |---> Training samples

Sentinel-2 2025 + WorldCover / validation |----> LULC prediction

WorldCover has 11 classes, while we are only focusin on focuses on:

###1. Water
###2. Vegetation
###3. Built-up
###4. Bare Land

So, remap WorldCover's classes into our four project classes.

####Tree cover-->	Vegetation
####Shrubland	-->Vegetation
####Grassland	--> Vegetation
####Cropland	--> Vegetation
####Mangroves	-->Vegetation
####Built-up	--> Built-up
####Bare / sparse vegetation-->	Bare Land
####Permanent water bodies--> Water
#####Herbaceous wetland	--> Water
####Moss / lichen	--> Vegetation

In [12]:
worldcover = ee.Image(
    'ESA/WorldCover/v100/2020'
).select('Map')

print("WorldCover loaded!")

WorldCover loaded!


In [13]:
worldcover_lulc = worldcover.remap(
    [
        10, 20, 30, 40,
        50,
        60,
        70,
        80,
        90,
        95,
        100
    ],
    [
        1, 1, 1, 1,
        2,
        3,
        99,
        0,
        0,
        1,
        1
    ]
).rename('LULC')

In [14]:
lulc_labels = worldcover_lulc.clip(study_area)

print("LULC reference map prepared!")

LULC reference map prepared!


In [15]:
lulc_vis = {
    'min': 0,
    'max': 3,
    'palette': [
        '0000FF',  # Water
        '00AA00',  # Vegetation
        'FF0000',  # Built-up
        'D2B48C'   # Bare land
    ]
}

m = geemap.Map()

m.centerObject(study_area, 10)

m.addLayer(
    lulc_labels,
    lulc_vis,
    'Reference LULC 2020'
)

m.addLayer(
    study_area,
    {},
    'Study Area'
)

m

Map(center=[17.425070303037895, 78.42499999999899], controls=(WidgetControl(options=['position', 'transparent_…

In [16]:
class_distribution = lulc_labels.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=study_area,
    scale=10,
    maxPixels=1e9
)

print(
    class_distribution.getInfo()
)

{'LULC': {'0': 192287.89019607843, '1': 11986540.68627444, '2': 5950832.6745098075, '3': 1387825.7764705885}}


In [17]:
training_image = features_2020.select(
    feature_bands
).addBands(
    lulc_labels
)

In [18]:
samples = training_image.stratifiedSample(
    numPoints=1000,
    classBand='LULC',
    region=study_area,
    scale=10,
    classValues=[0, 1, 2, 3],
    classPoints=[1000, 1000, 1000, 1000],
    geometries=False,
    seed=42
)

print(
    "Number of samples:",
    samples.size().getInfo()
)

Number of samples: 4000


In [19]:
samples_info = samples.getInfo()

rows = [
    feature['properties']
    for feature in samples_info['features']
]

df = pd.DataFrame(rows)

print(df.head())
print("\nDataset shape:", df.shape)

       B11      B12       B2       B3       B4       B8  LULC      NDBI  \
0  0.29570  0.25845  0.13640  0.17190  0.20890  0.25480     0  0.074296   
1  0.01150  0.00830  0.04030  0.06630  0.04340  0.03060     0 -0.453682   
2  0.09960  0.06140  0.06970  0.07465  0.06635  0.10545     0 -0.028530   
3  0.13110  0.07120  0.06185  0.07680  0.06635  0.16370     0 -0.110583   
4  0.01045  0.00760  0.06405  0.09385  0.05795  0.06025     0 -0.704385   

       NDVI      NDWI  
0  0.098986 -0.194282  
1 -0.172973  0.368421  
2  0.227590 -0.171016  
3  0.423169 -0.361331  
4  0.019459  0.218040  

Dataset shape: (4000, 10)


In [20]:
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna()

df = df[df['LULC'].isin([0, 1, 2, 3])]

print("Clean dataset shape:", df.shape)

Clean dataset shape: (4000, 10)


In [21]:
print(
    df['LULC'].value_counts().sort_index()
)

LULC
0    1000
1    1000
2    1000
3    1000
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split

X = df[feature_bands]

y = df['LULC']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 3200
Testing samples: 800


In [23]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_predictions = rf_model.predict(X_test)

In [24]:
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_precision = precision_score(
    y_test,
    rf_predictions,
    average='weighted',
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_predictions,
    average='weighted',
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_predictions,
    average='weighted',
    zero_division=0
)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)

Random Forest Results
---------------------
Accuracy : 0.75375
Precision: 0.7509698846088293
Recall   : 0.75375
F1 Score : 0.7507535041465513


In [25]:
print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=[
            'Water',
            'Vegetation',
            'Built-up',
            'Bare Land'
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

       Water       0.93      0.91      0.92       200
  Vegetation       0.73      0.79      0.76       200
    Built-up       0.72      0.79      0.75       200
   Bare Land       0.62      0.53      0.57       200

    accuracy                           0.75       800
   macro avg       0.75      0.75      0.75       800
weighted avg       0.75      0.75      0.75       800



In [26]:
import os

os.makedirs(
    '../models',
    exist_ok=True
)

joblib.dump(
    rf_model,
    '../models/random_forest_lulc.pkl'
)

print("Random Forest model saved!")

Random Forest model saved!


In [27]:
svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale'
)

svm_model.fit(
    X_train,
    y_train
)

svm_predictions = svm_model.predict(X_test)

print("SVM training completed!")

SVM training completed!


In [28]:
svm_accuracy = accuracy_score(
    y_test,
    svm_predictions
)

svm_precision = precision_score(
    y_test,
    svm_predictions,
    average='weighted',
    zero_division=0
)

svm_recall = recall_score(
    y_test,
    svm_predictions,
    average='weighted',
    zero_division=0
)

svm_f1 = f1_score(
    y_test,
    svm_predictions,
    average='weighted',
    zero_division=0
)

print("SVM Results")
print("---------------------")
print("Accuracy :", svm_accuracy)
print("Precision:", svm_precision)
print("Recall   :", svm_recall)
print("F1 Score :", svm_f1)

SVM Results
---------------------
Accuracy : 0.77125
Precision: 0.770536929608504
Recall   : 0.77125
F1 Score : 0.7662953498533014


In [29]:
print(
    classification_report(
        y_test,
        svm_predictions,
        target_names=[
            'Water',
            'Vegetation',
            'Built-up',
            'Bare Land'
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

       Water       0.94      0.90      0.92       200
  Vegetation       0.73      0.85      0.79       200
    Built-up       0.73      0.81      0.77       200
   Bare Land       0.68      0.52      0.59       200

    accuracy                           0.77       800
   macro avg       0.77      0.77      0.77       800
weighted avg       0.77      0.77      0.77       800



In [30]:
joblib.dump(
    svm_model,
    '../models/svm_lulc.pkl'
)

print("SVM model saved!")

SVM model saved!


In [31]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_predictions = xgb_model.predict(X_test)

print("XGBoost training completed!")

XGBoost training completed!


In [32]:
xgb_accuracy = accuracy_score(
    y_test,
    xgb_predictions
)

xgb_precision = precision_score(
    y_test,
    xgb_predictions,
    average='weighted',
    zero_division=0
)

xgb_recall = recall_score(
    y_test,
    xgb_predictions,
    average='weighted',
    zero_division=0
)

xgb_f1 = f1_score(
    y_test,
    xgb_predictions,
    average='weighted',
    zero_division=0
)

print("XGBoost Results")
print("---------------------")
print("Accuracy :", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall   :", xgb_recall)
print("F1 Score :", xgb_f1)

XGBoost Results
---------------------
Accuracy : 0.75875
Precision: 0.7547437418574353
Recall   : 0.75875
F1 Score : 0.7540389222490615


In [33]:
print(
    classification_report(
        y_test,
        xgb_predictions,
        target_names=[
            'Water',
            'Vegetation',
            'Built-up',
            'Bare Land'
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

       Water       0.92      0.92      0.92       200
  Vegetation       0.75      0.81      0.78       200
    Built-up       0.70      0.79      0.74       200
   Bare Land       0.64      0.51      0.57       200

    accuracy                           0.76       800
   macro avg       0.75      0.76      0.75       800
weighted avg       0.75      0.76      0.75       800



In [34]:
joblib.dump(
    xgb_model,
    '../models/xgboost_lulc.pkl'
)

print("XGBoost model saved!")

XGBoost model saved!


In [35]:
baseline_results = pd.DataFrame({
    'Model': [
        'Random Forest',
        'SVM',
        'XGBoost'
    ],
    'Accuracy': [
        rf_accuracy,
        svm_accuracy,
        xgb_accuracy
    ],
    'Precision': [
        rf_precision,
        svm_precision,
        xgb_precision
    ],
    'Recall': [
        rf_recall,
        svm_recall,
        xgb_recall
    ],
    'F1 Score': [
        rf_f1,
        svm_f1,
        xgb_f1
    ]
})

baseline_results

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.75375,0.750970,0.75375,0.750754
1,SVM,0.77125,0.770537,0.77125,0.766295
2,XGBoost,0.75875,0.754744,0.75875,0.754039


In [36]:
baseline_results.to_csv(
    '../models/baseline_results.csv',
    index=False
)

print("Baseline results saved!")

Baseline results saved!
